<a href="https://colab.research.google.com/github/Grashch/Work/blob/main/%D0%9E%D0%B1%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%BA%D0%B0_%D0%A0%D0%B0%D0%BC%D0%B0%D0%BD%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import pandas as pd

def average_txt_rows(
    input_file: str,
    output_file: str = None,
    n: int = 3,
    sep: str = None,
    header: int = None,
    skiprows: int = 0,
    output_sep: str = None,
    write_header: bool = False
) -> pd.DataFrame:
    """
    Усредняет каждые n строк в txt-файле с табличными данными.

    Параметры:
        input_file   : путь к входному txt-файлу
        output_file  : путь для сохранения результата (txt). Если None – вывод в консоль.
        n            : количество строк для усреднения (по умолчанию 3)
        sep          : разделитель столбцов (если None, определяется автоматически)
        header       : номер строки с заголовками (None – заголовков нет)
        skiprows     : количество пропускаемых строк в начале
        output_sep   : разделитель в выходном файле (по умолчанию такой же, как входной)
        write_header : записывать ли заголовки в выходной файл (по умолчанию False)

    Возвращает:
        pd.DataFrame с усреднёнными значениями.
    """
    # Автоопределение разделителя, если не задан
    if sep is None:
        with open(input_file, 'r', encoding='utf-8') as f:
            first_line = f.readline()
        if '\t' in first_line:
            sep = '\t'
        elif ',' in first_line:
            sep = ','
        else:
            sep = r'\s+'  # пробелы любой длины

    # Чтение данных
    df = pd.read_csv(
        input_file,
        sep=sep,
        header=header,
        skiprows=skiprows,
        engine='python'
    )

    if df.empty:
        raise ValueError("Файл пуст или не содержит данных.")

    # Создаём группу из каждых n строк
    df['_group'] = df.index // n

    # Усредняем по группам (только числовые столбцы)
    result = df.groupby('_group', as_index=False).mean(numeric_only=True)
    result.drop(columns=['_group'], inplace=True)
    result.reset_index(drop=True, inplace=True)

    # Сохранение или вывод
    if output_file:
        if output_sep is None:
            out_sep = sep
        else:
            out_sep = output_sep
        result.to_csv(
            output_file,
            sep=out_sep,
            index=False,
            header=write_header,   # <-- теперь управляем заголовками
            float_format='%.6f'
        )
        print(f"Результат сохранён в '{output_file}' с разделителем {repr(out_sep)}")
    else:
        print(result)

    return result

# ======== ПРИМЕР ИСПОЛЬЗОВАНИЯ ========
if __name__ == "__main__":
    average_txt_rows(
        input_file='data.txt',
        output_file='averaged_data.txt',
        n=3,
        # Если в исходном файле нет заголовков, header=None (по умолчанию)
        # Для сохранения без первой строки с номерами столбцов write_header=False
    )

Результат сохранён в 'averaged_data.txt' с разделителем '\t'
